# UIT_SGS_Foodseekers_C1 — XLM-RoBERTa (Lượt 3)
**Mô hình chọn (approved list):** `FacebookAI/xlm-roberta-base`

Phân loại 3 lớp `BENIGN / JAILBREAK / PROMPT_INJECTION` trên pt/fr/it/en, chấm Macro F1.

Cách chạy trên Colab: **Runtime → Change runtime type → T4 GPU**, rồi **Runtime → Run all**. Xong sẽ tự tải `submission.csv` về máy.

In [ ]:
# 1. Cài thư viện (Colab đã có torch/transformers, chỉ cần cập nhật nhẹ)
!pip -q install -U transformers accelerate
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU - hãy bật T4!")

In [ ]:
# 2. Cấu hình
MODEL_NAME = "FacebookAI/xlm-roberta-base"   # có thể đổi sang "FacebookAI/xlm-roberta-large" nếu GPU đủ VRAM
MAX_LEN    = 256
EPOCHS     = 3
LR         = 2e-5
BATCH      = 16
SEED       = 42
BASE_URL   = "https://raw.githubusercontent.com/bersious/Foodseekers/main/data"
LABELS     = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]
L2I = {l: i for i, l in enumerate(LABELS)}

In [ ]:
# 3. Tải dữ liệu từ repo của đội
import pandas as pd, numpy as np
train = pd.read_csv(f"{BASE_URL}/train.csv")
test  = pd.read_csv(f"{BASE_URL}/test.csv")
train["text"] = train["text"].fillna("").astype(str)
test["text"]  = test["text"].fillna("").astype(str)
print(train.shape, test.shape)
print(train["label"].value_counts(normalize=True).round(3))

In [ ]:
# 4. Chia train/validation (giữ nguyên tỉ lệ nhãn x ngôn ngữ)
from sklearn.model_selection import train_test_split
strat = train["label"] + "_" + train["language"]
tr_df, va_df = train_test_split(train, test_size=0.1, stratify=strat, random_state=SEED)
y_tr = tr_df["label"].map(L2I).values
y_va = va_df["label"].map(L2I).values
print(len(tr_df), len(va_df))

In [ ]:
# 5. Tokenize
from transformers import AutoTokenizer
from torch.utils.data import Dataset

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

class DS(Dataset):
    def __init__(self, texts, labels=None):
        self.enc = tok(list(texts), truncation=True, max_length=MAX_LEN)
        self.labels = labels
    def __len__(self): return len(self.enc["input_ids"])
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        if self.labels is not None: item["labels"] = int(self.labels[i])
        return item

ds_tr, ds_va, ds_te = DS(tr_df["text"], y_tr), DS(va_df["text"], y_va), DS(test["text"])

In [ ]:
# 6. Trainer với loss có trọng số lớp (xử lý mất cân bằng 62/31/7)
import torch.nn as nn
from sklearn.metrics import f1_score
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding, set_seed)
set_seed(SEED)

counts = np.bincount(y_tr, minlength=3)
class_w = torch.tensor(len(y_tr) / (3 * counts), dtype=torch.float)
print("Class weights:", class_w.numpy().round(3))

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=class_w.to(out.logits.device))(out.logits, labels)
        return (loss, out) if return_outputs else loss

def metrics(p):
    return {"macro_f1": f1_score(p.label_ids, p.predictions.argmax(-1), average="macro")}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3,
    id2label={i: l for i, l in enumerate(LABELS)}, label2id=L2I)

args = TrainingArguments(
    output_dir="out", num_train_epochs=EPOCHS, learning_rate=LR,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH * 2,
    warmup_ratio=0.1, weight_decay=0.01, fp16=torch.cuda.is_available(),
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    logging_steps=100, report_to="none", seed=SEED)

trainer = WeightedTrainer(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
                          data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
trainer.train()

In [ ]:
# 7. Đánh giá validation + tinh chỉnh hệ số nhân cho lớp hiếm
from scipy.special import softmax
from sklearn.metrics import classification_report

va_prob = softmax(trainer.predict(ds_va).predictions, axis=1)
print("Macro F1 (argmax):", round(f1_score(y_va, va_prob.argmax(1), average="macro"), 4))

best_w, best_f1 = (1.0, 1.0), -1
for wj in np.arange(0.8, 1.31, 0.05):
    for wp in np.arange(0.8, 2.01, 0.05):
        f = f1_score(y_va, (va_prob * [1.0, wj, wp]).argmax(1), average="macro")
        if f > best_f1: best_f1, best_w = f, (wj, wp)
print(f"Hệ số tốt nhất JAILBREAK x{best_w[0]:.2f}, PROMPT_INJECTION x{best_w[1]:.2f} -> Macro F1 = {best_f1:.4f}")
print(classification_report(y_va, (va_prob * [1.0, *best_w]).argmax(1), target_names=LABELS, digits=4))

# F1 theo từng ngôn ngữ
for lang in ["pt", "fr", "it", "en"]:
    m = (va_df["language"] == lang).values
    print(lang, round(f1_score(y_va[m], (va_prob[m] * [1.0, *best_w]).argmax(1), average="macro"), 4))

In [ ]:
# 8. Dự đoán test + kiểm tra file nộp
te_prob = softmax(trainer.predict(ds_te).predictions, axis=1)
pred = (te_prob * [1.0, *best_w]).argmax(1)
sub = pd.DataFrame({"id": test["id"], "label": [LABELS[i] for i in pred]})

assert len(sub) == 6000, "Sai số dòng!"
assert sub["label"].isin(LABELS).all() and sub["label"].notna().all()
assert sub["id"].is_unique

sub.to_csv("submission.csv", index=False)
np.save("xlmr_test_proba.npy", te_prob)   # giữ xác suất để ensemble với TF-IDF sau này
print(sub["label"].value_counts())

In [ ]:
# 9. Tải file về máy
try:
    from google.colab import files
    files.download("submission.csv")
    files.download("xlmr_test_proba.npy")
except ImportError:
    print("Không phải Colab - file nằm ở thư mục hiện tại.")